# Market / Volatility / News Sentiment LoRA Fine-tuning

This notebook fine-tunes the **Market + Volatility + News Sentiment Agent** on Colab using:

`data/gold/market_sentiment_finetune_data.jsonl`

It is configured for a free/low-memory Colab GPU by default using `Qwen/Qwen2.5-1.5B-Instruct`.

Before running:
1. In Colab: **Runtime -> Change runtime type -> T4 GPU**.
2. Upload your project folder to Google Drive, for example: `/content/drive/MyDrive/298A`.
3. Make sure these files exist in Drive:
   - `agents/finetune_lora.py`
   - `agents/infer_lora.py`
   - `data/gold/market_sentiment_finetune_data.jsonl`


## 1. Mount Drive And Configure Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Change this if your Drive folder name is different.
PROJECT_DIR = '/content/drive/MyDrive/298A'

from pathlib import Path
project = Path(PROJECT_DIR)

FINETUNE_SCRIPT = project / 'agents' / 'finetune_lora.py'
INFER_SCRIPT = project / 'agents' / 'infer_lora.py'
TRAIN_JSONL = project / 'data' / 'gold' / 'market_sentiment_finetune_data.jsonl'
OUTPUT_DIR = project / 'models' / 'market_sentiment_lora'

print('PROJECT_DIR:', project)
print('FINETUNE_SCRIPT:', FINETUNE_SCRIPT, FINETUNE_SCRIPT.exists())
print('INFER_SCRIPT:', INFER_SCRIPT, INFER_SCRIPT.exists())
print('TRAIN_JSONL:', TRAIN_JSONL, TRAIN_JSONL.exists())
print('OUTPUT_DIR:', OUTPUT_DIR)


## 2. Install Fine-tuning Dependencies

This installs the LoRA/QLoRA stack. Colab usually already has CUDA PyTorch installed.

In [ ]:
!pip install -q -U transformers datasets peft accelerate bitsandbytes sentencepiece protobuf


## 3. Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')


## 4. Validate Training JSONL

In [ ]:
import json
from collections import Counter

rows = []
with open(TRAIN_JSONL, encoding='utf-8') as f:
    for line in f:
        if line.strip():
            item = json.loads(line)
            rows.append(item)

labels = Counter()
for item in rows:
    assistant = json.loads(item['messages'][-1]['content'])
    labels[assistant['market_sentiment_risk_label']] += 1

print('examples:', len(rows))
print('label counts:', labels)
print('sample user:', json.loads(rows[0]['messages'][1]['content']))
print('sample assistant:', json.loads(rows[0]['messages'][2]['content']))


## 5. Choose Base Model

Default is a smaller non-gated model that fits Colab better. You can switch to Mistral later if you have access and enough GPU memory.

In [ ]:
BASE_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'

# Heavier option, only if you accepted the HF license and have enough GPU:
# BASE_MODEL = 'mistralai/Mistral-7B-Instruct-v0.3'

print('BASE_MODEL:', BASE_MODEL)


## 6. Optional Hugging Face Login

Qwen usually does not require login. If you switch to a gated model like Mistral, run this cell and paste your HF token.

In [ ]:
# Optional. Uncomment only if your selected model requires Hugging Face auth.
# from huggingface_hub import login
# login()


## 7. Train Market/News LoRA Adapter

For free Colab, this uses:
- 4-bit QLoRA automatically when CUDA is available
- short max length to reduce memory
- 2 epochs by default

If training is too slow, set `--epochs 1`.

In [ ]:
!python3 "{FINETUNE_SCRIPT}" \
  --train-jsonl "{TRAIN_JSONL}" \
  --base-model "{BASE_MODEL}" \
  --output-dir "{OUTPUT_DIR}" \
  --max-length 1024 \
  --epochs 2 \
  --batch-size 1 \
  --grad-accum 8 \
  --lr 2e-4


## 8. Verify Adapter Files

In [ ]:
!ls -lh "{OUTPUT_DIR}"
!test -f "{OUTPUT_DIR}/adapter_config.json" && echo 'adapter_config.json found'


## 9. Quick Inference Test

This confirms the saved adapter can load with the same base model.

In [ ]:
PROMPT = '{"ticker":"AAPL","task":"Assess market volatility and financial news sentiment risk.","features":{"annualized_volatility":0.28,"beta":1.12,"max_drawdown":-0.17,"sentiment_mean":0.03,"article_count":40}}'

!python3 "{INFER_SCRIPT}" \
  --base-model "{BASE_MODEL}" \
  --adapter-dir "{OUTPUT_DIR}" \
  --prompt "{PROMPT}" \
  --max-new-tokens 220


## 10. Zip Adapter For Download Or Sharing

In [ ]:
ADAPTER_ZIP = project / 'models' / 'market_sentiment_lora.zip'
!cd "{OUTPUT_DIR.parent}" && zip -qr market_sentiment_lora.zip market_sentiment_lora
print('Saved:', ADAPTER_ZIP)
